# ADR Evaluation (clean, fast, consistent)

This notebook is a simpler replacement for `ADR_Eval.ipynb`.

It does:
1) **Rollout evaluation** (absolute + relative errors), batched.
2) **Intrinsic metrics** along trajectories:
   - Dynamics Jacobian singular values (∂f/∂z)
   - Decoder gain along trajectories (approx spectral norm of J_dec)
   - Latent error (z_pred vs z_true = enc(u_true))
3) **Paired comparisons** between methods (same windows).

**Consistency rules**:
- `umin/umax` are computed on **training μ and training-time** (coarse) and reused everywhere.
- Evaluation defaults to **fine** (`u_fine`, `t_fine`) but can be switched.

In [1]:
import os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.dataset import norm_u, inv_norm_u, H5Snapshots
from src.node import integrate_latent_eval

In [2]:
experiments = {
    "vanilla": {
        "lam_reg": 0
    },
    "stoch_iso": {
        "lam_reg": 0.1
    },    
    "operator_norm": {
        "lam_reg": 0.1
    },
    "curvature": {
        "lam_reg": 1.0,
    },
    "stiefel": {
        "lam_reg": 0
    }
}

In [3]:
# -----------------
# USER CONFIG
# -----------------
H5_PATH    = "adr_full.h5" 
SPLIT      = "train"           # "train" | "interp" | "extrap"
EVAL_FIELD = "u_fine"           # "u_fine" (recommended) or "u_coarse"
RK_METHOD  = "rk2"              # "rk2" or "rk4"

HORIZON    = 320                # in indices of EVAL_FIELD grid
N_MU       = 100
N_STARTS   = 10
BATCH      = 32

V          = 1                 # AE version
SELECTION  = "best_swa"        # ode_{selection}.pt   best_swa | target_swa | last_swa
METHODS    = [x for x in experiments.keys()]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float32

# intrinsic metric cost controls
INTR_STEPS = 6
INTR_MAX_WINDOWS = 80
DEC_GAIN_ITERS = 5

SEED = 0
rng = np.random.default_rng(SEED)

print("DEVICE:", DEVICE)

DEVICE: cuda


In [4]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import copy

# --------- CONFIG ----------
TOP_K = 3

# If you want to restrict versions/runs/methods, set these; otherwise we auto-discover.
AE_VERSIONS = [2]
NODE_RUNS   =  range(1,16+1) # range(1,16+1)

# "Robust worst-best" target selection:
# q = 1.0  -> max(best_vals)  (strict "worst best")
# q = 0.9  -> 90th percentile (more robust to one outlier)
TARGET_Q = 1.0

OUTPUT_ROOT = Path("output/NODE")
MAP_LOCATION = "cuda"


In [5]:
def parse_method_version_run(stats_path: Path):
    # stats_path: output/NODE/{method}_v{version}/run_{run}/stats.json
    run_dir = stats_path.parent
    method_v = run_dir.parent.name          # "{method}_v{version}"
    run_name = run_dir.name                 # "run_{run}"

    if "_v" not in method_v:
        raise ValueError(f"Cannot parse version from: {method_v}")
    method, v_str = method_v.rsplit("_v", 1)
    version = int(v_str)

    if not run_name.startswith("run_"):
        raise ValueError(f"Cannot parse run from: {run_name}")
    run = int(run_name.split("_", 1)[1])

    return method, version, run, run_dir

# Discover all stats.json files
stats_files = sorted(OUTPUT_ROOT.glob("*_v*/run_*/stats.json"))
print("Found stats.json:", len(stats_files))

records = []
stats_map = {}  # (version, method, run) -> dict(stats) + run_dir

for p in stats_files:
    method, version, run, run_dir = parse_method_version_run(p)

    if AE_VERSIONS is not None and version not in AE_VERSIONS:
        continue
    if NODE_RUNS is not None and run not in NODE_RUNS:
        continue

    with open(p, "r") as f:
        st = json.load(f)

    # basic sanity
    if "mse_val" not in st or "mse_train" not in st:
        print(f"[skip] missing mse_train/mse_val in {p}")
        continue

    stats_map[(version, method, run)] = {"stats": st, "run_dir": run_dir}

print("Loaded runs:", len(stats_map))
# list(stats_map.keys())


Found stats.json: 160
Loaded runs: 5


In [6]:
best_vals = []
for (version, method, run), obj in stats_map.items():
    val = np.asarray(obj["stats"]["mse_val"], dtype=float)
    best_vals.append(val.min())

best_vals = np.asarray(best_vals, dtype=float)
if len(best_vals) == 0:
    raise RuntimeError("No stats loaded; check OUTPUT_ROOT or filters.")

target_val = float(np.quantile(best_vals, TARGET_Q))
print(f"Target val MSE: (q={TARGET_Q}): {target_val:.6e}")
print(f"Best-val range: {float(best_vals.min()):.6e} to {float(best_vals.max()):.6e}")


Target val MSE: (q=1.0): 1.754809e-03
Best-val range: 4.679813e-04 to 1.754809e-03


In [7]:
def first_epoch_runningbest_leq(val_mse: np.ndarray, target: float):
    """
    Return 1-based epoch where running best first reaches <= target.
    If never reaches, return None.
    """
    rb = np.minimum.accumulate(val_mse)
    hits = np.where(rb <= target)[0]
    if len(hits) == 0:
        return None
    return int(hits[0] + 1)

def trailing_epochs(ep: int, k: int):
    """Return [max(1, ep-k+1), ..., ep] (1-based)."""
    start = max(1, ep - k + 1)
    return list(range(start, ep + 1))

def load_ckpt_model(run_dir: Path, epoch: int):
    ckpt_path = run_dir / f"ode_{epoch}.pt"
    if not ckpt_path.exists():
        return None, ckpt_path
    model = torch.load(ckpt_path, map_location=MAP_LOCATION, weights_only=False)
    model.eval()
    return model, ckpt_path

def swa_average_checkpoints(run_dir: Path, epochs: list[int], save_path: Path):
    """
    Uniform average of floating tensors over checkpoints in `epochs`.
    Non-floating tensors are copied from the LAST loaded checkpoint.
    Saves a torch model at save_path.
    Returns: (used_epochs, save_path) or ([], None) if nothing loaded.
    """
    used = []
    sum_sd = None
    last_sd = None
    template_model = None

    for ep in epochs:
        model, ckpt_path = load_ckpt_model(run_dir, ep)
        if model is None:
            print(f"[warn] missing {ckpt_path}")
            continue

        sd = model.state_dict()
        if sum_sd is None:
            template_model = copy.deepcopy(model)
            sum_sd = {k: v.detach().clone().to(MAP_LOCATION) for k, v in sd.items()}
        else:
            for k in sum_sd:
                if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
                    sum_sd[k].add_(sd[k].detach().to(MAP_LOCATION, dtype=sum_sd[k].dtype))
                else:
                    # keep placeholder; we'll overwrite from last checkpoint below
                    pass

        last_sd = {k: v.detach().to(MAP_LOCATION) for k, v in sd.items()}
        used.append(ep)

    if not used:
        print(f"[skip] no checkpoints found for {run_dir}")
        return [], None

    # divide floating tensors
    n = float(len(used))
    for k in sum_sd:
        if torch.is_floating_point(sum_sd[k]) or torch.is_complex(sum_sd[k]):
            sum_sd[k].div_(n)
        else:
            # copy non-floating from last checkpoint
            sum_sd[k] = last_sd[k].clone()

    template_model.load_state_dict(sum_sd, strict=True)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(template_model, save_path)
    return used, save_path


In [8]:
rows = []

for (version, method, run), obj in sorted(stats_map.items()):
    st = obj["stats"]
    run_dir: Path = obj["run_dir"]

    train = np.asarray(st["mse_train"], dtype=float)
    val   = np.asarray(st["mse_val"], dtype=float)
    n_epochs = len(val)

    # epochs (1-based)
    best_train_ep = int(np.argmin(train) + 1)
    best_val_ep   = int(np.argmin(val) + 1)

    # "target reached" epoch using running best
    target_ep = first_epoch_runningbest_leq(val, target_val)
    target_reached = target_ep is not None
    if target_ep is None:
        target_ep = n_epochs  # fallback: use last epoch if never reached

    # trailing windows
    epochs_target = trailing_epochs(target_ep, TOP_K)
    epochs_best   = trailing_epochs(best_val_ep, TOP_K)
    epochs_last   = trailing_epochs(n_epochs, TOP_K)

    # save SWA models
    used_t, p_t = swa_average_checkpoints(run_dir, epochs_target, run_dir / "ode_target_swa.pt")
    used_b, p_b = swa_average_checkpoints(run_dir, epochs_best,   run_dir / "ode_best_swa.pt")
    used_l, p_l = swa_average_checkpoints(run_dir, epochs_last,   run_dir / "ode_last_swa.pt")

    rows.append({
        "version": version,
        "method": method,
        "run": run,
        "n_epochs": n_epochs,

        "best_train": float(train.min()),
        "best_val": float(val.min()),
        "best_train_ep": best_train_ep,
        "best_val_ep": best_val_ep,

        "target_val": float(target_val),
        "target_ep": int(target_ep),
        "target_reached": bool(target_reached),

        "std_train": float(train.std(ddof=0)),
        "std_val": float(val.std(ddof=0)),

        "target_swa_epochs": used_t,
        "best_swa_epochs": used_b,
        "last_swa_epochs": used_l,
    })

metadata_df = pd.DataFrame(rows).sort_values(["version", "method", "run"]).reset_index(drop=True)
metadata_df


,version,method,run,n_epochs,best_train,best_val,best_train_ep,best_val_ep,target_val,target_ep,target_reached,std_train,std_val,target_swa_epochs,best_swa_epochs,last_swa_epochs
0,2,curvature,1,50,0.001021,0.000699,48,50,0.001755,21,True,0.024411,0.022872,"[19, 20, 21]","[48, 49, 50]","[48, 49, 50]"
1,2,operator_norm,1,50,0.001171,0.001755,48,28,0.001755,28,True,0.024841,0.023479,"[26, 27, 28]","[26, 27, 28]","[48, 49, 50]"
2,2,stiefel,1,50,0.000655,0.000489,50,46,0.001755,12,True,0.021354,0.011239,"[10, 11, 12]","[44, 45, 46]","[48, 49, 50]"
3,2,stoch_iso,1,50,0.000945,0.001082,50,41,0.001755,23,True,0.024374,0.023157,"[21, 22, 23]","[39, 40, 41]","[48, 49, 50]"
4,2,vanilla,1,50,0.000627,0.000468,50,49,0.001755,15,True,0.021344,0.013012,"[13, 14, 15]","[47, 48, 49]","[48, 49, 50]"


In [9]:
# Summary across runs: mean/std of key quantities per (version, method)
agg_cols = [
    "best_train", "best_val",
    "best_train_ep", "best_val_ep",
    "target_ep", "std_train", "std_val"
]

summary_df = (
    metadata_df
    .groupby(["version", "method"], as_index=False)[agg_cols]
    .agg(["mean", "std", "count"])
)

# flatten MultiIndex columns
summary_df.columns = ["version", "method"] + [f"{c0}_{c1}" for (c0, c1) in summary_df.columns[2:]]
summary_df = summary_df.sort_values(["version", "best_val_mean"]).reset_index(drop=True)
summary_df


,version,method,best_train_mean,best_train_std,best_train_count,best_val_mean,best_val_std,best_val_count,best_train_ep_mean,best_train_ep_std,...,best_val_ep_count,target_ep_mean,target_ep_std,target_ep_count,std_train_mean,std_train_std,std_train_count,std_val_mean,std_val_std,std_val_count
0,2,vanilla,0.000627,NaN,1,0.000468,NaN,1,50.0,NaN,...,1,15.0,NaN,1,0.021344,NaN,1,0.013012,NaN,1
1,2,stiefel,0.000655,NaN,1,0.000489,NaN,1,50.0,NaN,...,1,12.0,NaN,1,0.021354,NaN,1,0.011239,NaN,1
2,2,curvature,0.001021,NaN,1,0.000699,NaN,1,48.0,NaN,...,1,21.0,NaN,1,0.024411,NaN,1,0.022872,NaN,1
3,2,stoch_iso,0.000945,NaN,1,0.001082,NaN,1,50.0,NaN,...,1,23.0,NaN,1,0.024374,NaN,1,0.023157,NaN,1
4,2,operator_norm,0.001171,NaN,1,0.001755,NaN,1,48.0,NaN,...,1,28.0,NaN,1,0.024841,NaN,1,0.023479,NaN,1


In [10]:
# -----------------
# Helpers: time grids + split segments from H5
# -----------------
def read_time_and_kind(f: h5py.File, field: str):
    if "coarse" in field:
        return f["t_coarse"][...], "coarse"
    return f["t_fine"][...], "fine"

def load_segment(f: h5py.File, kind: str, split: str):
    # Prefer explicit split datasets from the paper-aligned generator
    if "splits" in f:
        if kind == "fine":
            key = {"train":"splits/fine_ids_train",
                   "interp":"splits/fine_ids_interp",
                   "extrap":"splits/fine_ids_extrap"}[split]
        else:
            key = {"train":"splits/coarse_ids_train",
                   "interp":"splits/coarse_ids_val_loop",
                   "extrap":"splits/coarse_ids_val_loop"}[split]
        ids = np.asarray(f[key][...], dtype=np.int64)
        return int(ids[0]), int(ids[-1])

    # Fallback: infer from attrs iT1_*, iT2_*
    if kind == "fine":
        iT1 = int(f.attrs["iT1_fine"]); iT2 = int(f.attrs["iT2_fine"]); Nt = f["t_fine"].shape[0]
    else:
        iT1 = int(f.attrs["iT1_coarse"]); iT2 = int(f.attrs["iT2_coarse"]); Nt = f["t_coarse"].shape[0]

    if split == "train":  return 0, iT1
    if split == "interp": return iT1 + 1, iT2
    if split == "extrap": return iT2 + 1, Nt - 1
    raise ValueError(split)

with h5py.File(H5_PATH, "r") as f:
    t_eval_np, kind = read_time_and_kind(f, EVAL_FIELD)
    seg0, seg1 = load_segment(f, kind, SPLIT)
    U = f[f"{SPLIT}/{EVAL_FIELD}"]
    Nmu_all, Nt_all, H, W = U.shape

print("EVAL_FIELD:", EVAL_FIELD, "kind:", kind, "U:", (Nmu_all, Nt_all, H, W))
print("Segment:", SPLIT, "t ids:", (seg0, seg1), "len:", (seg1 - seg0 + 1))

EVAL_FIELD: u_fine kind: fine U: (1000, 1001, 32, 32)
Segment: train t ids: (0, 400) len: 401


In [11]:
# -----------------
# Normalization: compute umin/umax on (train μ) x (train-time) using u_coarse.
# Cached to norm_cache.json.
# -----------------
def get_or_compute_norm(h5_path: str, cache_path: str = "norm_cache.json"):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as g:
            d = json.load(g)
        if d.get("h5_path") == os.path.abspath(h5_path):
            return float(d["umin"]), float(d["umax"])

    with h5py.File(h5_path, "r") as f:
        train_idx = np.asarray(f["train_idx"][...], dtype=np.int64)
        if "splits" in f and "coarse_ids_train" in f["splits"]:
            train_t_ids = np.asarray(f["splits/coarse_ids_train"][...], dtype=np.int64)
        else:
            iT1 = int(f.attrs["iT1_coarse"])
            train_t_ids = np.arange(0, iT1 + 1, dtype=np.int64)

    ds = H5Snapshots(h5_path, split="train", field="u_coarse", mu_indices=train_idx, return_mu=False)
    umin, umax = ds.compute_train_minmax(field="u_coarse", mu_ids=train_idx, t_ids=train_t_ids, split="train")

    with open(cache_path, "w") as g:
        json.dump({"h5_path": os.path.abspath(h5_path), "umin": float(umin), "umax": float(umax)}, g)

    return float(umin), float(umax)

umin, umax = get_or_compute_norm(H5_PATH)
print("umin/umax:", umin, umax)

umin/umax: 0.0 3.20582914352417


In [12]:
# -----------------
# Sample evaluation windows (paired across methods)
# -----------------
def sample_windows(h5_path: str, split: str, field: str, horizon: int, n_mu: int, n_starts: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    with h5py.File(h5_path, "r") as f:
        U = f[f"{split}/{field}"]
        Nmu, Nt, *_ = U.shape
        t_np, kind = read_time_and_kind(f, field)
        seg0, seg1 = load_segment(f, kind, split)

    max_k0 = seg1 - horizon
    if max_k0 < seg0:
        raise ValueError(f"Segment too short for horizon={horizon}: seg=[{seg0},{seg1}]")

    mu_ids = rng.choice(np.arange(Nmu), size=min(n_mu, Nmu), replace=True)
    windows = []
    for mu in mu_ids:
        k0s = rng.integers(low=seg0, high=max_k0 + 1, size=n_starts)
        for k0 in k0s:
            windows.append((int(mu), int(k0)))
    return windows

windows = sample_windows(H5_PATH, SPLIT, EVAL_FIELD, HORIZON, N_MU, N_STARTS, seed=SEED)
print("Num windows:", len(windows), "example:", windows[:5])

Num windows: 1000 example: [(850, 32), (850, 63), (850, 25), (850, 19), (850, 64)]


In [13]:
# -----------------
# Load models
# -----------------
def load_models(methods, V: int, run: int, selection: str, device: str):
    out = {}
    for m in methods:
        enc = torch.load(f"output/AE_v{V}/{m}/enc_target.pt", map_location=device, weights_only=False)
        dec = torch.load(f"output/AE_v{V}/{m}/dec_target.pt", map_location=device, weights_only=False)
        ode = torch.load(f"output/NODE/{m}_v{V}/run_{run}/ode_{selection}.pt", map_location=device, weights_only=False)
        enc.eval(); dec.eval(); ode.eval()
        out[m] = (enc.to(device), dec.to(device), ode.to(device))
    return out



In [14]:
# -----------------
# Batched rollout + error metrics
# -----------------
def get_mu_tensor(mu_all_np, mu_ids, device):
    # mu_all_np: (Nmu, 3) float32 numpy
    return torch.as_tensor(mu_all_np[mu_ids], device=device, dtype=DTYPE)

def gather_u_windows(f: h5py.File, split: str, field: str, batch_windows, horizon: int):
    U = f[f"{split}/{field}"]
    out = []
    for (mu, k0) in batch_windows:
        out.append(np.asarray(U[mu, k0:k0+horizon+1], dtype=np.float32))  # (L+1,H,W)
    return np.stack(out, axis=0)  # (B,L+1,H,W)

def build_time_grid(t_np: np.ndarray, k0s: np.ndarray, horizon: int, device: str):
    """
    Exact absolute-time grid per window, required because the ODE uses absolute-time encoding.
    Returns t_grid: (B, L+1) torch float32.
    """
    offsets = np.arange(horizon + 1, dtype=np.int64)[None, :]  # (1,L+1)
    idx = k0s[:, None] + offsets                                # (B,L+1)
    t_grid_np = t_np[idx]                                       # (B,L+1)
    return torch.as_tensor(t_grid_np, device=device, dtype=DTYPE)

@torch.no_grad()
def rollout(enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method="rk2", drop_t0=True):
    """
    u_win_np: (B,L+1,H,W) physical
    t_grid:   (B,L+1) absolute time
    mu_t:     (B,nmu)
    """
    B, Lp1, H, W = u_win_np.shape
    device = mu_t.device

    u_true = torch.as_tensor(u_win_np, device=device, dtype=DTYPE).unsqueeze(2)  # (B,L+1,1,H,W)
    u0 = u_true[:, 0]  # (B,1,H,W)

    u0_n = norm_u(u0, umin, umax)
    z0 = enc(u0_n)

    z_pred = integrate_latent_eval(ode, z0, t_grid, mu_t, method=rk_method)   # (B,L+1,1,4,4)
    u_hat_n = dec(z_pred.reshape(-1, *z_pred.shape[2:])).reshape(B, Lp1, 1, H, W)
    u_hat = inv_norm_u(u_hat_n, umin, umax)

    # errors (physical space)
    diff = (u_hat - u_true).reshape(B, Lp1, -1)
    tru  = (u_true).reshape(B, Lp1, -1)

    num = torch.linalg.vector_norm(diff, dim=-1)  # (B,L+1)
    den = torch.linalg.vector_norm(tru,  dim=-1)  # (B,L+1)

    if drop_t0:
        num = num[:, 1:]
        den = den[:, 1:]
        diff = diff[:, 1:]

    rel = num / (den + 1e-8)
    mse = diff.pow(2).mean(dim=-1)

    metrics = {
        "abs_mean": num.mean(dim=1),
        "abs_max":  num.max(dim=1).values,
        "abs_fin":  num[:, -1],
        "rel_mean": rel.mean(dim=1),
        "rel_max":  rel.max(dim=1).values,
        "rel_fin":  rel[:, -1],
        "mse_fin":  mse[:, -1],
    }
    return metrics, z_pred, u_hat, u_true

In [15]:
# -----------------
# Intrinsic metrics
# -----------------
def dyn_jac_svals(ode, z, t_scalar, mu_row):
    with torch.enable_grad():
        z_flat = z.reshape(-1).detach().requires_grad_(True)

        def f_flat(z_flat_):
            zz = z_flat_.view(1, 1, 4, 4)
            out = ode(t_scalar.view(1), zz, mu_row.view(1, -1))
            return out.reshape(-1)

        J = torch.autograd.functional.jacobian(f_flat, z_flat, create_graph=False)
        return torch.linalg.svdvals(J).detach()


def decoder_gain_power(dec, z, n_iter=5):
    """Approx spectral norm of J_dec at z via power iteration on J^T J."""
    with torch.enable_grad():
        v = torch.randn_like(z)
        v = v / (v.flatten().norm() + 1e-12)

        for _ in range(n_iter):
            z0 = z.detach().requires_grad_(True)

            # y has a grad graph wrt z0
            y = dec(z0).flatten()

            # Jv as a *constant* vector (no need for create_graph=True)
            _, jv = torch.autograd.functional.jvp(
                lambda zz: dec(zz).flatten(),
                (z0,),
                (v,),
                create_graph=False
            )

            # grad wrt z0 gives J^T (Jv)
            dot = (y * jv.detach()).sum()
            jt_jv = torch.autograd.grad(dot, z0, retain_graph=False, create_graph=False)[0]

            v = jt_jv.detach()
            v = v / (v.flatten().norm() + 1e-12)

        # final sigma estimate = ||J v||
        z0 = z.detach().requires_grad_(True)
        _, jv = torch.autograd.functional.jvp(
            lambda zz: dec(zz).flatten(),
            (z0,),
            (v,),
            create_graph=False
        )
        return jv.norm().detach()


@torch.no_grad()
def latent_truth(enc, u_true, umin, umax):
    B, Lp1 = u_true.shape[0], u_true.shape[1]
    u_n = norm_u(u_true.reshape(-1, *u_true.shape[2:]), umin, umax)
    z_true = enc(u_n).reshape(B, Lp1, 1, 4, 4)
    return z_true

def intrinsic_for_batch(enc, dec, ode, z_pred, u_true, t_grid, mu_t, steps_idx, dec_iters=5):
    rows = []
    z_true = latent_truth(enc, u_true, umin, umax)

    B = z_pred.shape[0]
    for b in range(B):
        mu_row = mu_t[b]
        for sidx in steps_idx:
            z = z_pred[b, sidx:sidx+1]           # (1,1,4,4)
            t_scalar = t_grid[b, sidx]

            svals = dyn_jac_svals(ode, z, t_scalar, mu_row)
            gain  = decoder_gain_power(dec, z, n_iter=dec_iters)
            zerr  = (z_pred[b, sidx] - z_true[b, sidx]).reshape(-1).norm()

            rows.append({
                "b": b,
                "step": int(sidx),
                "t": float(t_scalar.detach().cpu()),
                "dyn_sigma_max": float(svals.max().cpu()),
                "dyn_sigma_min": float(svals.min().cpu()),
                "dyn_cond": float((svals.max()/(svals.min()+1e-12)).cpu()),
                "dec_gain": float(gain.cpu()),
                "latent_err": float(zerr.detach().cpu()),
            })
    return pd.DataFrame(rows)

In [16]:
# -----------------
# Evaluate all methods (paired windows)
# -----------------
def eval_all(models, windows, split, field, horizon, batch_size):
    all_rows = []
    intr_rows = []

    steps_idx = np.linspace(0, horizon, num=INTR_STEPS).round().astype(int).tolist()
    intr_windows = set(windows[:min(INTR_MAX_WINDOWS, len(windows))])

    with h5py.File(H5_PATH, "r") as f:
        t_np, _ = read_time_and_kind(f, field)

        MU_ALL = None
        mu_key = f"{split}/mu"
        if mu_key in f:
            MU_ALL = np.asarray(f[mu_key][...], dtype=np.float32)

        for method, (enc, dec, ode) in models.items():
            print("Evaluating:", method)

            for i0 in range(0, len(windows), batch_size):
                batch_windows = windows[i0:i0+batch_size]
                mu_ids = np.asarray([w[0] for w in batch_windows], dtype=np.int64)
                k0s    = np.asarray([w[1] for w in batch_windows], dtype=np.int64)

                u_win_np = gather_u_windows(f, split, field, batch_windows, horizon)
                t_grid = build_time_grid(t_np, k0s, horizon, DEVICE)

                mu_t = (
                    get_mu_tensor(MU_ALL, mu_ids, DEVICE)
                    if MU_ALL is not None
                    else torch.zeros((len(mu_ids), 0), device=DEVICE, dtype=DTYPE)
                )

                metrics, z_pred, u_hat, u_true = rollout(
                    enc, dec, ode, u_win_np, t_grid, mu_t, umin, umax, rk_method=RK_METHOD
                )

                for j, (mu_id, k0) in enumerate(batch_windows):
                    all_rows.append({
                        "method": method,
                        "split": split,
                        "field": field,
                        "horizon": horizon,
                        "mu_id": int(mu_id),
                        "k0": int(k0),
                        "abs_mean": float(metrics["abs_mean"][j].cpu()),
                        "abs_max":  float(metrics["abs_max"][j].cpu()),
                        "abs_fin":  float(metrics["abs_fin"][j].cpu()),
                        "rel_mean": float(metrics["rel_mean"][j].cpu()),
                        "rel_max":  float(metrics["rel_max"][j].cpu()),
                        "rel_fin":  float(metrics["rel_fin"][j].cpu()),
                        "mse_fin":  float(metrics["mse_fin"][j].cpu()),
                    })

                if any((w in intr_windows) for w in batch_windows):
                    keep = [idx for idx,w in enumerate(batch_windows) if w in intr_windows]
                    if keep:
                        df_intr = intrinsic_for_batch(
                            enc, dec, ode,
                            z_pred[keep], u_true[keep], t_grid[keep], mu_t[keep],
                            steps_idx=steps_idx, dec_iters=DEC_GAIN_ITERS
                        )
                        df_intr["method"] = method
                        df_intr["split"] = split
                        df_intr["field"] = field
                        df_intr["horizon"] = horizon
                        df_intr["mu_id"] = df_intr["b"].map(lambda bb: int(batch_windows[keep[bb]][0]))
                        df_intr["k0"]    = df_intr["b"].map(lambda bb: int(batch_windows[keep[bb]][1]))
                        df_intr.drop(columns=["b"], inplace=True)
                        intr_rows.append(df_intr)

    df_roll = pd.DataFrame(all_rows)
    df_intr = pd.concat(intr_rows, ignore_index=True) if intr_rows else pd.DataFrame()
    return df_roll, df_intr


# -----------------
# Summary tables
# -----------------
def summarize(df, cols):
    g = df.groupby("method")[cols]
    out = g.agg(["mean","std","count"])
    for c in cols:
        out[(c,"se")] = out[(c,"std")] / np.sqrt(out[(c,"count")].clip(lower=1))
    return out.sort_values((cols[0],"mean"))


# -----------------
# Paired comparisons vs vanilla
# -----------------
from scipy.stats import wilcoxon

def paired_vs_baseline(df, metric: str, baseline: str = "vanilla"):
    keys = ["mu_id","k0"]
    wide = df.pivot_table(index=keys, columns="method", values=metric, aggfunc="first")
    if baseline not in wide.columns:
        raise ValueError(f"baseline {baseline} not present in {list(wide.columns)}")
    deltas = wide.sub(wide[baseline], axis=0).drop(columns=[baseline], errors="ignore")

    rows=[]
    for m in deltas.columns:
        x = deltas[m].dropna().to_numpy()
        if len(x) >= 5:
            p_less = wilcoxon(x, alternative="less").pvalue     # delta < 0 is better
            p_two  = wilcoxon(x, alternative="two-sided").pvalue
        else:
            p_less = p_two = np.nan
        rows.append({
            "metric": metric,
            "method": m,
            "n": len(x),
            "delta_mean": float(np.mean(x)) if len(x) else np.nan,
            "delta_median": float(np.median(x)) if len(x) else np.nan,
            "p_less": p_less,
            "p_two_sided": p_two,
        })
    return pd.DataFrame(rows).sort_values("p_less")

    
for seed in NODE_RUNS:
    print(f"RUN {seed} [{SELECTION}] ----------------------------------------")

    models = load_models(METHODS, V, seed, SELECTION, DEVICE)
    # print("Loaded:", list(models.keys()))

    df_roll, df_intr = eval_all(models, windows, SPLIT, EVAL_FIELD, HORIZON, BATCH)
    print(df_roll.head())
    print("roll rows:", len(df_roll), "intr rows:", len(df_intr))

    summary = summarize(df_roll, ["rel_mean","rel_max","rel_fin","abs_fin","mse_fin"])
    print(f"[{seed}] summary:")
    print(summary)

    
    paired = paired_vs_baseline(df_roll, "rel_mean")
    print(f"{seed} [{SELECTION}] paired:")
    print(paired)

    os.makedirs(f"stats/{SELECTION}", exist_ok=True)

    df_roll.to_csv(f"stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv")
    df_intr.to_csv(f"stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv")  

    summary.to_csv(f"stats/{SELECTION}/summary_H{HORIZON}_v{V}_{seed}.csv")

    paired.to_csv(f"stats/{SELECTION}/paired_H{HORIZON}_v{V}_{seed}.csv")

    print("----------------------------------------\n")

RUN 1 [best_swa] ----------------------------------------
Evaluating: vanilla
Evaluating: stoch_iso
Evaluating: operator_norm
Evaluating: curvature
Evaluating: stiefel
    method  split   field  horizon  mu_id  k0  abs_mean   abs_max   abs_fin  \
0  vanilla  train  u_fine      320    850  32  2.478064  3.323443  2.390381   
1  vanilla  train  u_fine      320    850  63  2.069670  2.924423  1.955427   
2  vanilla  train  u_fine      320    850  25  2.577741  3.440584  2.584568   
3  vanilla  train  u_fine      320    850  19  2.731169  3.495948  2.547561   
4  vanilla  train  u_fine      320    850  64  2.077308  2.933281  1.954892   

   rel_mean   rel_max   rel_fin   mse_fin  
0  0.057900  0.079704  0.064576  0.005580  
1  0.049227  0.069483  0.051747  0.003734  
2  0.060012  0.081007  0.067678  0.006523  
3  0.063449  0.083535  0.064487  0.006338  
4  0.049480  0.069382  0.051611  0.003732  
roll rows: 5000 intr rows: 2400
[1] summary:
               rel_mean                   rel_ma

In [17]:
df_intr = pd.DataFrame()
df_roll = pd.DataFrame()


for seed in range(1,4+1):
    df_i = pd.read_csv(f"stats/{SELECTION}/df_intr_H{HORIZON}{V}_{seed}.csv") 
    df_r = pd.read_csv(f"stats/{SELECTION}/df_roll_H{HORIZON}_v{V}_{seed}.csv")
    
    df_intr = pd.concat([df_intr,df_i])
    df_roll = pd.concat([df_roll,df_r])



In [18]:
df_intr.head(10)

,Unnamed: 0,step,t,dyn_sigma_max,dyn_sigma_min,dyn_cond,dec_gain,latent_err,method,split,field,horizon,mu_id,k0
0,0,0,1.005310,4.483844,0.023757,188.735397,3.472589,0.000100,vanilla,train,u_fine,320,850,32
1,1,64,3.015929,2.398250,0.020863,114.952812,3.281405,0.585143,vanilla,train,u_fine,320,850,32
2,2,128,5.026548,2.062143,0.003662,563.063599,3.377689,0.581823,vanilla,train,u_fine,320,850,32
3,3,192,7.037168,4.560758,0.049159,92.775375,3.440066,0.869052,vanilla,train,u_fine,320,850,32
4,4,256,9.047787,3.296242,0.000795,4147.690430,3.519788,0.641024,vanilla,train,u_fine,320,850,32
5,5,320,11.058406,2.597236,0.006024,431.124268,3.361383,0.557689,vanilla,train,u_fine,320,850,32
6,6,0,1.979203,4.494231,0.019204,234.031647,2.985576,0.000135,vanilla,train,u_fine,320,850,63
7,7,64,3.989823,2.405667,0.045333,53.067062,3.424309,0.438447,vanilla,train,u_fine,320,850,63
8,8,128,6.000442,2.795761,0.020931,133.567902,2.823895,0.481890,vanilla,train,u_fine,320,850,63
9,9,192,8.011062,3.130514,0.030405,102.961594,3.502879,0.615287,vanilla,train,u_fine,320,850,63


In [19]:
# -----------------
# Intrinsic metric summaries
# -----------------
if len(df_intr):
    intr_summary = df_intr.groupby("method")[["dyn_sigma_max","dyn_cond","dec_gain","latent_err"]].agg(["mean","std","count"])
    intr_summary
else:
    print("No intrinsic rows computed.")

In [20]:
intr_summary

dyn_sigma_max                     dyn_cond                      \
                       mean       std count         mean           std count   
method                                                                         
curvature          4.383055  1.149500  1920  1624.816590  12895.870627  1920   
operator_norm      4.639447  1.154835  1920  2867.885376  36829.609615  1920   
stiefel            3.391973  0.861266  1920   653.022260   3746.105601  1920   
stoch_iso          4.613838  1.159159  1920  2050.513720  14384.831424  1920   
vanilla            3.077164  0.738650  1920   624.410176   2151.353368  1920   

               dec_gain                 latent_err                  
                   mean       std count       mean       std count  
method                                                              
curvature      1.577663  0.080522  1920   2.830723  2.252235  1920  
operator_norm  1.025095  0.023423  1920   3.301712  2.981975  1920  
stiefel        4.625306  0.226934  1920   0.801638  0.594340  1920  
stoch_iso      1.000242  0.001144  1920   3.294474  3.056107  1920  
vanilla        3.313416  0.193040  1920   0.981035  0.714762  1920

In [21]:
# intr_summary.to_csv(f"stats/{SELECTION}/intr_summary_H{HORIZON}_v{V}_{RUN}.csv")

In [22]:
import numpy as np
import pandas as pd

df = df_intr.copy()
df = df.drop(columns=[c for c in ["Unnamed: 0"] if c in df.columns])

# Make sure numeric
num_cols = ["step","t","dyn_sigma_max","dyn_sigma_min","dyn_cond","dec_gain","latent_err"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Logs (clip avoids -inf if something hits 0)
eps = 1e-12
df["log10_dyn_cond"] = np.log10(np.clip(df["dyn_cond"].to_numpy(), eps, None))
df["log10_sigma_max"] = np.log10(np.clip(df["dyn_sigma_max"].to_numpy(), eps, None))
df["log10_sigma_min"] = np.log10(np.clip(df["dyn_sigma_min"].to_numpy(), eps, None))
df["log10_dec_gain"]  = np.log10(np.clip(df["dec_gain"].to_numpy(), eps, None))

# Tail / “catastrophe” rates (tune thresholds after looking at quantiles)
df["cond_gt_1e4"]  = df["dyn_cond"] > 1e4
df["cond_gt_1e5"]  = df["dyn_cond"] > 1e5
df["sigmax_gt_10"] = df["dyn_sigma_max"] > 10


In [23]:
qs = [0.5, 0.9, 0.95, 0.99, 0.999]

def qcols(s):
    q = s.quantile(qs)
    q.index = [f"q{int(p*1000):03d}" for p in qs]  # q500 q900 q950 q990 q999
    return q

summary_method = (
    df.groupby("method")
      .apply(lambda g: pd.concat([
          qcols(g["log10_dyn_cond"]).add_prefix("log10_dyn_cond_"),
          qcols(g["latent_err"]).add_prefix("latent_err_"),
          qcols(g["dyn_sigma_max"]).add_prefix("sigmax_"),
          g[["cond_gt_1e4","cond_gt_1e5","sigmax_gt_10"]].mean().add_prefix("rate_"),
      ]))
      .reset_index()
)

summary_method


,method,log10_dyn_cond_q500,log10_dyn_cond_q900,log10_dyn_cond_q950,log10_dyn_cond_q990,log10_dyn_cond_q999,latent_err_q500,latent_err_q900,latent_err_q950,latent_err_q990,latent_err_q999,sigmax_q500,sigmax_q900,sigmax_q950,sigmax_q990,sigmax_q999,rate_cond_gt_1e4,rate_cond_gt_1e5,rate_sigmax_gt_10
0,curvature,2.530132,3.243958,3.510065,4.427662,5.084327,2.577294,5.873973,7.085796,9.011993,10.151958,4.226971,5.866253,6.356753,7.543690,10.288213,0.015625,0.001563,0.002083
1,operator_norm,2.647680,3.407546,3.735168,4.378817,5.362659,2.561852,7.306527,8.923036,12.900752,15.444105,4.553338,6.197388,6.686300,7.565349,9.123546,0.030729,0.003125,0.000000
2,stiefel,2.180463,2.910788,3.245630,3.947943,4.849667,0.774747,1.634614,1.921478,2.340656,2.916709,3.278374,4.575435,4.962224,5.788478,6.738298,0.007812,0.000000,0.000000
3,stoch_iso,2.640706,3.423537,3.705053,4.314142,5.393300,2.743160,7.225253,8.865620,13.591927,20.082563,4.467023,6.158948,6.775821,7.736094,9.086582,0.023958,0.002604,0.000521
4,vanilla,2.312985,3.030474,3.276576,3.884166,4.512065,0.894022,1.906354,2.329284,3.106234,3.909331,3.004964,4.031810,4.396297,5.209226,5.894514,0.007812,0.000000,0.000000


In [24]:
run_keys = ["method","split","field","horizon","mu_id","k0"]

run_stats = (
    df.groupby(run_keys)
      .agg(
          logcond_med=("log10_dyn_cond","median"),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          cond_rate_1e5=("cond_gt_1e5","mean"),
          sigmax_med=("dyn_sigma_max","median"),
          laterr_med=("latent_err","median"),
      )
      .reset_index()
)

# Aggregate across runs (now each run is one row)
method_run_summary = (
    run_stats.groupby(["method","split","field","horizon"])
             .agg(["mean","std","median"])
)

method_run_summary


mu_id                     \
                                           mean         std median   
method        split  field  horizon                                  
curvature     extrap u_fine 320       68.126582   57.276739   61.0   
              train  u_fine 320      349.467532  285.868418  307.0   
operator_norm extrap u_fine 320       68.126582   57.276739   61.0   
              train  u_fine 320      349.467532  285.868418  307.0   
stiefel       extrap u_fine 320       68.126582   57.276739   61.0   
              train  u_fine 320      349.467532  285.868418  307.0   
stoch_iso     extrap u_fine 320       68.126582   57.276739   61.0   
              train  u_fine 320      349.467532  285.868418  307.0   
vanilla       extrap u_fine 320       68.126582   57.276739   61.0   
              train  u_fine 320      349.467532  285.868418  307.0   

                                             k0                   logcond_med  \
                                           mean        std median        mean   
method        split  field  horizon                                             
curvature     extrap u_fine 320      595.898734  53.047051  596.0    2.533711   
              train  u_fine 320       42.389610  24.037884   42.0    2.544966   
operator_norm extrap u_fine 320      595.898734  53.047051  596.0    2.650631   
              train  u_fine 320       42.389610  24.037884   42.0    2.718775   
stiefel       extrap u_fine 320      595.898734  53.047051  596.0    2.211751   
              train  u_fine 320       42.389610  24.037884   42.0    2.115871   
stoch_iso     extrap u_fine 320      595.898734  53.047051  596.0    2.674840   
              train  u_fine 320       42.389610  24.037884   42.0    2.620195   
vanilla       extrap u_fine 320      595.898734  53.047051  596.0    2.305652   
              train  u_fine 320       42.389610  24.037884   42.0    2.363273   

                                                        logcond_p99  ...  \
                                          std    median        mean  ...   
method        split  field  horizon                                  ...   
curvature     extrap u_fine 320      0.123378  2.527828    3.623811  ...   
              train  u_fine 320      0.201717  2.557418    3.394015  ...   
operator_norm extrap u_fine 320      0.154769  2.632150    3.794213  ...   
              train  u_fine 320      0.216733  2.690433    3.571972  ...   
stiefel       extrap u_fine 320      0.107140  2.197300    3.413701  ...   
              train  u_fine 320      0.201574  2.113077    2.854693  ...   
stoch_iso     extrap u_fine 320      0.132836  2.664854    3.880632  ...   
              train  u_fine 320      0.185044  2.584409    3.383875  ...   
vanilla       extrap u_fine 320      0.139976  2.304609    3.363408  ...   
              train  u_fine 320      0.198045  2.327822    3.138018  ...   

                                              cond_rate_1e5                   \
                                       median          mean       std median   
method        split  field  horizon                                            
curvature     extrap u_fine 320      3.456138      0.000703  0.006250    0.0   
              train  u_fine 320      3.293159      0.004329  0.026683    0.0   
operator_norm extrap u_fine 320      3.729522      0.002110  0.010686    0.0   
              train  u_fine 320      3.409699      0.006494  0.032462    0.0   
stiefel       extrap u_fine 320      3.398125      0.000000  0.000000    0.0   
              train  u_fine 320      2.802262      0.000000  0.000000    0.0   
stoch_iso     extrap u_fine 320      3.747763      0.003516  0.013613    0.0   
              train  u_fine 320      3.361829      0.000000  0.000000    0.0   
vanilla       extrap u_fine 320      3.382068      0.000000  0.000000    0.0   
              train  u_fine 320      3.043528      0.000000  0.000000    0.0   

                                    sigmax_med 

In [25]:
time_profile = (
    df.groupby(["method","split","field","horizon","step"])
      .agg(
          logcond_med=("log10_dyn_cond","median"),
          logcond_p95=("log10_dyn_cond", lambda s: s.quantile(0.95)),
          logcond_p99=("log10_dyn_cond", lambda s: s.quantile(0.99)),
          n=("log10_dyn_cond","size"),
      )
      .reset_index()
)

time_profile


,method,split,field,horizon,step,logcond_med,logcond_p95,logcond_p99,n
0,curvature,extrap,u_fine,320,0,2.572643,3.497274,4.382916,240
1,curvature,extrap,u_fine,320,64,2.561640,3.473991,4.245050,240
2,curvature,extrap,u_fine,320,128,2.551083,3.559366,4.162136,240
3,curvature,extrap,u_fine,320,192,2.493298,3.304664,3.688936,240
4,curvature,extrap,u_fine,320,256,2.480912,3.477332,3.913928,240
5,curvature,extrap,u_fine,320,320,2.503952,3.448099,4.609360,240
6,curvature,train,u_fine,320,0,2.586471,3.535173,4.270657,80
7,curvature,train,u_fine,320,64,2.383272,3.455736,4.942230,80
8,curvature,train,u_fine,320,128,2.612024,3.840812,5.037373,80
9,curvature,train,u_fine,320,192,2.537230,3.700383,4.432654,80


In [26]:
# roll_df must contain: method, split, field, horizon, mu_id, k0, rel_mean (or mse_fin, etc.)
intr_per_run = run_stats.copy()

merged = intr_per_run.merge(
    df_roll[["method","split","field","horizon","mu_id","k0","rel_mean","mse_fin"]],
    on=["method","split","field","horizon","mu_id","k0"],
    how="inner",
)

merged[["logcond_med","logcond_p99","cond_rate_1e5","laterr_med","rel_mean","mse_fin"]].corr()


,logcond_med,logcond_p99,cond_rate_1e5,laterr_med,rel_mean,mse_fin
logcond_med,1.000000,0.442420,0.150123,0.447510,0.128193,0.102136
logcond_p99,0.442420,1.000000,0.375570,0.364096,0.265357,0.190660
cond_rate_1e5,0.150123,0.375570,1.000000,0.062117,0.013690,0.052791
laterr_med,0.447510,0.364096,0.062117,1.000000,0.804567,0.596952
rel_mean,0.128193,0.265357,0.013690,0.804567,1.000000,0.657506
mse_fin,0.102136,0.190660,0.052791,0.596952,0.657506,1.000000
